# ChatGPT Conversation History - Conversation Patterns & Dynamics
markdown
markdown
## Data structure overview
Quick checks of columns, types, missing values, and basic derivations to ensure downstream cells run cleanly.
code
python
# Inspect dataframe structure and try light derivations
import json
print('COLUMNS AND DTYPES')
print(df.dtypes)
print('







































---These cells add clustering, temporal shifts, topic–engagement matrices, coherence heuristics, and an outlier detector.## Next-layer Analyses (added)markdownmarkdownprint('
Data-structure checks complete — ready for next-layer analyses')    print('
No `topic` column found')else:    print(df['topic'].value_counts().head(10))    print(f"
Unique topics: {df['topic'].nunique()} (top 10)")if 'topic' in df.columns:# Topic overviewprint(df[num_cols].describe().T.round(2))print('
NUMERIC SUMMARY (describe)')num_cols = df.select_dtypes(include=[np.number]).columns.tolist()# Numeric summary for quick sanity    print('Converted `create_date` to datetime (coerced invalids to NaT)')    df['create_date'] = pd.to_datetime(df['create_date'], errors='coerce')if 'create_date' in df.columns and not np.issubdtype(df['create_date'].dtype, np.datetime64):# Ensure create_date is datetime            print('Could not derive `message_count` from `messages`:', e)        except Exception as e:            print('Derived `message_count` from `messages` column')            df['message_count'] = df['messages'].apply(lambda x: len(x) if isinstance(x, (list, tuple)) else (len(json.loads(x)) if isinstance(x, str) else 0))        try:    if 'messages' in df.columns:if 'message_count' not in df.columns:# Try to derive message_count when missing        print(f'Missing column: {c}')    if c not in df.columns:for c in required:            'ratio','complexity_score','engagement_score','topic','topic_confidence','title','create_date']required = ['message_count','user_messages','assistant_messages','tool_messages',# Required columns to check    display(df.head(5))with pd.option_context('display.max_columns', None):print('
SAMPLE ROWS (first 5)')print(missing[missing > 0].sort_values(ascending=False))missing = df.isnull().sum()MISSING VALUE COUNTS (non-zero only)')

## Setup

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
from collections import Counter
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")
%matplotlib inline

print("Libraries loaded!")

In [ ]:
# Load data
import os
if not os.path.exists('conversations.json'):
    raise FileNotFoundError("conversations.json not found — put it in the repo root or update the path.")
with open('conversations.json', 'r', encoding='utf-8') as f:
    conversations = json.load(f)

df = pd.read_csv('../chatgpt_conversations_with_topics.csv')
df['create_date'] = pd.to_datetime(df['create_date'])

print(f"Loaded {len(conversations):,} conversations")

## 1. Message Exchange Patterns

In [ ]:
# Analyze user-assistant ratio distribution
print("USER-ASSISTANT INTERACTION RATIOS")
print("=" * 60)
print(f"Mean ratio: {df['ratio'].mean():.2f}")
print(f"Median ratio: {df['ratio'].median():.2f}")
print(f"Std dev: {df['ratio'].std():.2f}")
print(f"\nPercentiles:")
for p in [25, 50, 75, 90, 95]:
    val = df['ratio'].quantile(p/100)
    print(f"  {p}th: {val:.2f}")

In [ ]:
# Visualize ratio distribution
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Histogram of ratios
axes[0, 0].hist(df['ratio'], bins=50, edgecolor='black', alpha=0.7)
axes[0, 0].axvline(df['ratio'].mean(), color='red', linestyle='--', linewidth=2, label=f'Mean: {df["ratio"].mean():.2f}')
axes[0, 0].axvline(df['ratio'].median(), color='green', linestyle='--', linewidth=2, label=f'Median: {df["ratio"].median():.2f}')
axes[0, 0].set_xlabel('Assistant/User Message Ratio')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('Distribution of Interaction Ratios', fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Scatter: user messages vs assistant messages
axes[0, 1].scatter(df['user_messages'], df['assistant_messages'], alpha=0.5, s=30)
# Add perfect balance line
max_val = max(df['user_messages'].max(), df['assistant_messages'].max())
axes[0, 1].plot([0, max_val], [0, max_val], 'r--', linewidth=2, label='1:1 ratio')
axes[0, 1].set_xlabel('User Messages')
axes[0, 1].set_ylabel('Assistant Messages')
axes[0, 1].set_title('User vs Assistant Messages', fontweight='bold')
axes[0, 1].legend()
axes[0, 1].grid(True, alpha=0.3)

# Conversation length vs ratio
axes[1, 0].scatter(df['message_count'], df['ratio'], alpha=0.5, s=30)
axes[1, 0].set_xlabel('Total Messages')
axes[1, 0].set_ylabel('Assistant/User Ratio')
axes[1, 0].set_title('Conversation Length vs Interaction Ratio', fontweight='bold')
axes[1, 0].grid(True, alpha=0.3)

# Tool message distribution
tool_msg_counts = df['tool_messages'].value_counts().sort_index().head(20)
axes[1, 1].bar(tool_msg_counts.index, tool_msg_counts.values, edgecolor='black', alpha=0.7)
axes[1, 1].set_xlabel('Tool Messages per Conversation')
axes[1, 1].set_ylabel('Frequency')
axes[1, 1].set_title('Tool Message Distribution', fontweight='bold')
axes[1, 1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

## 2. Conversation Complexity Analysis

In [ ]:
# Define complexity score
# Higher = more complex (longer, more messages, more tool usage)
df['complexity_score'] = (
    (df['message_count'] / df['message_count'].max()) * 0.4 +
    (df['ratio'] / df['ratio'].max()) * 0.3 +
    (df['tool_messages'] / df['tool_messages'].max()) * 0.3
)

# Categorize conversations by complexity
df['complexity_category'] = pd.cut(df['complexity_score'], 
                                   bins=[0, 0.25, 0.5, 0.75, 1.0],
                                   labels=['Low', 'Medium', 'High', 'Very High'])

complexity_dist = df['complexity_category'].value_counts().sort_index()

print("CONVERSATION COMPLEXITY DISTRIBUTION")
print("=" * 60)
for category, count in complexity_dist.items():
    pct = (count / len(df)) * 100
    print(f"{category:<15} {count:>4} conversations ({pct:>5.1f}%)")
print("=" * 60)

In [ ]:
# Visualize complexity
fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# Distribution
axes[0].hist(df['complexity_score'], bins=50, edgecolor='black', alpha=0.7, color='purple')
axes[0].set_xlabel('Complexity Score')
axes[0].set_ylabel('Frequency')
axes[0].set_title('Conversation Complexity Distribution', fontweight='bold')
axes[0].grid(True, alpha=0.3)

# Category pie chart
colors_complexity = ['#90EE90', '#FFD700', '#FF8C00', '#DC143C']
axes[1].pie(complexity_dist.values, labels=complexity_dist.index,
           autopct='%1.1f%%', colors=colors_complexity, startangle=90)
axes[1].set_title('Complexity Categories', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Show most and least complex conversations
print("\nTOP 10 MOST COMPLEX CONVERSATIONS")
print("=" * 100)
top_complex = df.nlargest(10, 'complexity_score')
for idx, row in top_complex.iterrows():
    title = row['title'][:70] + '...' if len(row['title']) > 70 else row['title']
    print(f"Score: {row['complexity_score']:.3f} | Msgs: {row['message_count']:<3} | Ratio: {row['ratio']:.2f} | Tools: {row['tool_messages']:<3} | {title}")

print("\n\nTOP 10 SIMPLEST CONVERSATIONS")
print("=" * 100)
least_complex = df.nsmallest(10, 'complexity_score')
for idx, row in least_complex.iterrows():
    title = row['title'][:70] + '...' if len(row['title']) > 70 else row['title']
    print(f"Score: {row['complexity_score']:.3f} | Msgs: {row['message_count']:<3} | Ratio: {row['ratio']:.2f} | Tools: {row['tool_messages']:<3} | {title}")

## 3. Message Flow Patterns

In [ ]:
# Analyze message sequences in conversations
def analyze_message_flow(conv):
    """Analyze the sequence of messages in a conversation."""
    roles = [msg['role'] for msg in conv['messages']]
    
    # Count transitions
    transitions = []
    for i in range(len(roles) - 1):
        transitions.append(f"{roles[i]} → {roles[i+1]}")
    
    return {
        'roles': roles,
        'transitions': transitions,
        'unique_roles': set(roles),
        'starts_with': roles[0] if roles else None,
        'ends_with': roles[-1] if roles else None
    }

# Analyze all conversations
all_transitions = []
start_roles = []
end_roles = []

for conv in conversations:
    flow = analyze_message_flow(conv)
    all_transitions.extend(flow['transitions'])
    if flow['starts_with']:
        start_roles.append(flow['starts_with'])
    if flow['ends_with']:
        end_roles.append(flow['ends_with'])

transition_counts = Counter(all_transitions)
start_counts = Counter(start_roles)
end_counts = Counter(end_roles)

print("MESSAGE FLOW ANALYSIS")
print("=" * 70)
print("\nTop 10 Message Transitions:")
for transition, count in transition_counts.most_common(10):
    pct = (count / len(all_transitions)) * 100
    print(f"  {transition:<30} {count:>6} ({pct:>5.1f}%)")

print("\nConversations Start With:")
for role, count in start_counts.most_common():
    pct = (count / len(start_roles)) * 100
    print(f"  {role:<15} {count:>5} ({pct:>5.1f}%)")

print("\nConversations End With:")
for role, count in end_counts.most_common():
    pct = (count / len(end_roles)) * 100
    print(f"  {role:<15} {count:>5} ({pct:>5.1f}%)")

In [ ]:
# Visualize message transitions
top_transitions = transition_counts.most_common(15)
trans_labels, trans_counts = zip(*top_transitions)

plt.figure(figsize=(12, 8))
bars = plt.barh(range(len(trans_labels)), trans_counts, edgecolor='black')

# Color by transition type
colors = ['green' if 'user' in t.lower() else 'blue' if 'assistant' in t.lower() else 'orange' 
          for t in trans_labels]
for bar, color in zip(bars, colors):
    bar.set_color(color)

plt.yticks(range(len(trans_labels)), trans_labels)
plt.xlabel('Frequency')
plt.title('Top 15 Message Transitions', fontsize=14, fontweight='bold')
plt.gca().invert_yaxis()
plt.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
plt.show()

## 4. Engagement Metrics

In [ ]:
# Calculate engagement score
# Based on: message count, back-and-forth ratio, tool usage
df['engagement_score'] = (
    np.log1p(df['message_count']) * 0.4 +
    (1 / (1 + abs(df['ratio'] - 1))) * 0.3 +  # Closer to 1:1 = more balanced
    np.log1p(df['tool_messages']) * 0.3
)

# Normalize to 0-100 scale
df['engagement_score'] = (df['engagement_score'] / df['engagement_score'].max()) * 100

print("ENGAGEMENT METRICS")
print("=" * 60)
print(f"Mean engagement score: {df['engagement_score'].mean():.1f}")
print(f"Median engagement score: {df['engagement_score'].median():.1f}")
print(f"\nEngagement Percentiles:")
for p in [25, 50, 75, 90, 95]:
    val = df['engagement_score'].quantile(p/100)
    print(f"  {p}th: {val:.1f}")

In [ ]:
# Visualize engagement
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Histogram
axes[0, 0].hist(df['engagement_score'], bins=50, edgecolor='black', alpha=0.7, color='teal')
axes[0, 0].axvline(df['engagement_score'].mean(), color='red', linestyle='--', linewidth=2, 
                  label=f'Mean: {df["engagement_score"].mean():.1f}')
axes[0, 0].set_xlabel('Engagement Score')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].set_title('Engagement Score Distribution', fontweight='bold')
axes[0, 0].legend()
axes[0, 0].grid(True, alpha=0.3)

# Engagement over time
df_time = df.sort_values('create_date')
df_time['month'] = df_time['create_date'].dt.to_period('M')
monthly_engagement = df_time.groupby('month')['engagement_score'].mean()
monthly_dates = [p.start_time for p in monthly_engagement.index]

axes[0, 1].plot(monthly_dates, monthly_engagement.values, marker='o', linewidth=2)
axes[0, 1].fill_between(monthly_dates, monthly_engagement.values, alpha=0.3)
axes[0, 1].set_xlabel('Month')
axes[0, 1].set_ylabel('Average Engagement Score')
axes[0, 1].set_title('Engagement Over Time', fontweight='bold')
axes[0, 1].grid(True, alpha=0.3)

# Engagement vs message count
axes[1, 0].scatter(df['message_count'], df['engagement_score'], alpha=0.5, s=30)
axes[1, 0].set_xlabel('Message Count')
axes[1, 0].set_ylabel('Engagement Score')
axes[1, 0].set_title('Engagement vs Conversation Length', fontweight='bold')
axes[1, 0].grid(True, alpha=0.3)

# Engagement by complexity category
engagement_by_complexity = df.groupby('complexity_category')['engagement_score'].mean()
axes[1, 1].bar(range(len(engagement_by_complexity)), engagement_by_complexity.values, 
              edgecolor='black', color=colors_complexity)
axes[1, 1].set_xticks(range(len(engagement_by_complexity)))
axes[1, 1].set_xticklabels(engagement_by_complexity.index)
axes[1, 1].set_ylabel('Average Engagement Score')
axes[1, 1].set_title('Engagement by Complexity', fontweight='bold')
axes[1, 1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

In [ ]:
# Most engaging conversations
print("\nTOP 15 MOST ENGAGING CONVERSATIONS")
print("=" * 100)
top_engaging = df.nlargest(15, 'engagement_score')
for idx, row in top_engaging.iterrows():
    title = row['title'][:70] + '...' if len(row['title']) > 70 else row['title']
    print(f"Score: {row['engagement_score']:>5.1f} | Msgs: {row['message_count']:<3} | {title}")

## 5. Correlation Analysis

In [ ]:
# Calculate correlations between different metrics
metrics = ['message_count', 'user_messages', 'assistant_messages', 'tool_messages',
          'ratio', 'complexity_score', 'engagement_score', 'topic_confidence']

correlation_matrix = df[metrics].corr()

print("CORRELATION ANALYSIS")
print("\nStrong Positive Correlations (> 0.5):")
for i in range(len(metrics)):
    for j in range(i+1, len(metrics)):
        corr = correlation_matrix.iloc[i, j]
        if corr > 0.5:
            print(f"  {metrics[i]:<25} ↔ {metrics[j]:<25} {corr:.3f}")

print("\nStrong Negative Correlations (< -0.5):")
for i in range(len(metrics)):
    for j in range(i+1, len(metrics)):
        corr = correlation_matrix.iloc[i, j]
        if corr < -0.5:
            print(f"  {metrics[i]:<25} ↔ {metrics[j]:<25} {corr:.3f}")

In [ ]:
# Visualize correlation matrix
plt.figure(figsize=(12, 10))
sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', 
           center=0, square=True, linewidths=1, cbar_kws={'label': 'Correlation'})
plt.title('Correlation Matrix of Conversation Metrics', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()

## 6. Pattern Discovery

In [ ]:
# Find conversations with unusual patterns
print("UNUSUAL CONVERSATION PATTERNS\n")

# Very high ratio (assistant talks much more than user)
high_ratio = df[df['ratio'] > df['ratio'].quantile(0.95)]
print(f"High Assistant-to-User Ratio (top 5%): {len(high_ratio)} conversations")
print("Examples:")
for idx, row in high_ratio.nlargest(5, 'ratio').iterrows():
    title = row['title'][:60] + '...' if len(row['title']) > 60 else row['title']
    print(f"  Ratio: {row['ratio']:.2f} | {title}")

# Very low ratio (more balanced or user talks more)
low_ratio = df[df['ratio'] < df['ratio'].quantile(0.05)]
print(f"\nLow Assistant-to-User Ratio (bottom 5%): {len(low_ratio)} conversations")
print("Examples:")
for idx, row in low_ratio.nsmallest(5, 'ratio').iterrows():
    title = row['title'][:60] + '...' if len(row['title']) > 60 else row['title']
    print(f"  Ratio: {row['ratio']:.2f} | {title}")

# High tool usage
high_tools = df[df['tool_messages'] > df['tool_messages'].quantile(0.95)]
print(f"\nHigh Tool Usage (top 5%): {len(high_tools)} conversations")
print("Examples:")
for idx, row in high_tools.nlargest(5, 'tool_messages').iterrows():
    title = row['title'][:60] + '...' if len(row['title']) > 60 else row['title']
    print(f"  Tools: {row['tool_messages']:<3} | {title}")

# Very long conversations
very_long = df[df['message_count'] > df['message_count'].quantile(0.95)]
print(f"\nVery Long Conversations (top 5%): {len(very_long)} conversations")
print("Examples:")
for idx, row in very_long.nlargest(5, 'message_count').iterrows():
    title = row['title'][:60] + '...' if len(row['title']) > 60 else row['title']
    print(f"  Messages: {row['message_count']:<3} | {title}")

## 7. Save Enhanced Dataset

In [ ]:
# Save enhanced dataframe with all calculated metrics
df.to_csv('../chatgpt_conversations_enhanced.csv', index=False)
print("✓ Saved enhanced DataFrame with all metrics to 'chatgpt_conversations_enhanced.csv'")

print("\nNew columns added:")
new_cols = ['complexity_score', 'complexity_category', 'engagement_score']
for col in new_cols:
    print(f"  • {col}")

## 8. Pattern Insights Summary

In [ ]:
print("\n" + "=" * 80)
print("CONVERSATION PATTERN INSIGHTS")
print("=" * 80)

print("\n💬 INTERACTION STYLE:")
avg_ratio = df['ratio'].mean()
print(f"  • Average assistant/user ratio: {avg_ratio:.2f}")
if avg_ratio > 1.3:
    print(f"  • You tend to ask complex questions requiring detailed responses")
elif avg_ratio > 0.9:
    print(f"  • Balanced back-and-forth conversation style")
else:
    print(f"  • Quick Q&A interaction pattern")

print("\n🔧 TOOL USAGE:")
tool_users = (df['tool_messages'] > 0).sum()
tool_pct = (tool_users / len(df)) * 100
print(f"  • {tool_pct:.1f}% of conversations use tools")
print(f"  • Average {df['tool_messages'].mean():.1f} tool messages per conversation")
if df['tool_messages'].mean() > 2:
    print(f"  • Heavy user of advanced ChatGPT features")

print("\n📊 COMPLEXITY:")
for category in ['Low', 'Medium', 'High', 'Very High']:
    count = complexity_dist.get(category, 0)
    pct = (count / len(df)) * 100
    print(f"  • {category}: {pct:.1f}%")

print("\n🎯 ENGAGEMENT:")
print(f"  • Average engagement score: {df['engagement_score'].mean():.1f}/100")
high_engagement = (df['engagement_score'] > 70).sum()
high_eng_pct = (high_engagement / len(df)) * 100
print(f"  • {high_eng_pct:.1f}% of conversations are highly engaging (score > 70)")

print("\n🔄 MESSAGE FLOW:")
most_common_trans = transition_counts.most_common(1)[0]
print(f"  • Most common transition: {most_common_trans[0]} ({most_common_trans[1]:,} times)")
print(f"  • {(start_counts.get('user', 0) / len(start_roles) * 100):.1f}% of conversations start with user")
print(f"  • {(end_counts.get('assistant', 0) / len(end_roles) * 100):.1f}% of conversations end with assistant")

print("\n" + "=" * 80)
print("Analysis Complete! All notebooks have generated comprehensive insights.")
print("=" * 80)

## Next-layer Analyses (added)
These cells add clustering, temporal shifts, topic–engagement matrices, coherence heuristics, and an outlier detector.

In [ ]:
# 1) Conversation archetypes (clustering)
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
features = ['message_count', 'user_messages', 'assistant_messages',
            'tool_messages', 'ratio', 'complexity_score', 'engagement_score']
X = df[features].fillna(0)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
# Try several k and inspect inertia / silhouette separately if you like
k = 4
kmeans = KMeans(n_clusters=k, random_state=42, n_init='auto')
df['conversation_cluster'] = kmeans.fit_predict(X_scaled)
print("CLUSTER PROFILES")
print("=
80
,
,
,

CLUSTER COUNTS")
print(df['conversation_cluster'].value_counts().sort_index())

In [ ]:
# 2) Temporal behavior shifts (monthly)
df_time = df.sort_values('create_date').copy()
df_time['year_month'] = df_time['create_date'].dt.to_period('M')
metrics_over_time = df_time.groupby('year_month')[[
    'message_count', 'ratio', 'tool_messages',
    'complexity_score', 'engagement_score'
]].mean().round(2)
print("METRICS OVER TIME (MONTHLY MEANS)")
print("=
80
,
,
for col in diff.columns:
    max_change = diff[col].abs().idxmax()
    print(f' • {col}: largest change at {max_change} = {diff.loc[max_change, col]}')

In [ ]:
# 3) Topic × engagement/complexity matrix
# Requires `topic` and `topic_confidence` (from with_topics CSV)
topic_cols = ['message_count', 'tool_messages',
              'complexity_score', 'engagement_score', 'topic_confidence']
topic_summary = (
    df.groupby('topic')[topic_cols]
      .agg(['count', 'mean'])
      .round(2)
      .sort_values(('engagement_score', 'mean'), ascending=False)
)
print("TOPIC–LEVEL PATTERNS")
print("=
80
,

: 
,
: {
: 

: [
4
,
0.25
0.75
,
0.25
0.75
,
0.25
0.75
,
,
,
RUNAWAY THREAD EXAMPLES (LONG, LOW ENGAGEMENT)")
print("=
80
,

In [ ]:
# 5) Outlier detector for 'weird' conversations
from sklearn.ensemble import IsolationForest
iso_features = ['message_count', 'tool_messages',
                'ratio', 'complexity_score', 'engagement_score']
X_iso = df[iso_features].fillna(0)
iso = IsolationForest(
    contamination=0.03,
    random_state=42
)
df['outlier_flag'] = iso.fit_predict(X_iso)  # -1 = outlier, 1 = normal
weird = df[df['outlier_flag'] == -1]
print(f"Detected {len(weird)} outlier conversations (~{len(weird)/len(df)*100:.1f}%).\n")
print("SAMPLE WEIRD CONVERSATIONS")
print("=
80
,